# Federal Funding EDA
Exploratory Data Analysis of federal spending data from USASpending.gov

## Section 1 — Setup & Data Loading

In [ ]:
# Cell 1 — Library Imports
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')

def billions(x, _):  return f'${x/1e9:.1f}B'
def trillions(x, _): return f'${x/1e12:.2f}T'

print('Libraries loaded successfully')

Standard imports — pandas and numpy for data work, matplotlib and seaborn for charts. The `billions` and `trillions` helper functions are just formatters so the y-axes on plots show `$1.2T` instead of raw scientific notation.

In [ ]:
# Cell 2 — File Paths (cleaned data)
BASE    = Path('..').resolve()
CLEAN_A = BASE / 'pipeline-a-hierarchical' / 'data' / 'cleaned'
CLEAN_B = BASE / 'pipeline-b-geography'    / 'data'

PATHS = {
    'budget_functions'   : CLEAN_A / 'budget_functions_quarterly_all.csv',
    'budget_subfunctions': CLEAN_A / 'budget_subfunctions_quarterly_all.csv',
    'functions_ref'      : CLEAN_A / 'budget_functions.csv',
    'subfunctions_ref'   : CLEAN_A / 'budget_subfunctions.csv',
    'agency'             : CLEAN_A / 'agency_ALL_FY.csv',
    'federal_accounts'   : CLEAN_A / 'federal_accounts_ALL_FY.csv',
    'geo_state'          : CLEAN_B / 'basic-geography'  / 'cleaned' / 'geography_state_all_FY2008_2024.csv',
    'geo_state_funding'  : CLEAN_B / 'agency-geography' / 'cleaned' / 'geo_state_funding_ALL_FY.csv',
    'geo_state_awarding' : CLEAN_B / 'agency-geography' / 'cleaned' / 'geo_state_awarding_ALL_FY.csv',
}

print('File paths configured:')
for name, path in PATHS.items():
    print(f'  {"OK" if path.exists() else "MISSING":7} {name:25} {path.name}')

All 9 files point to the `cleaned/` folders — these are the outputs from `00_data_cleaning.ipynb`, not the raw CSVs. Any file showing `MISSING` means the cleaning notebook hasn't been run yet for that file.

In [ ]:
# Cell 3 — Load All 9 DataFrames
dfs = {}
for name, path in PATHS.items():
    dfs[name] = pd.read_csv(path, dtype={'budget_function_id': str, 'budget_subfunction_id': str})
    print(f'Loaded  {name:25}  {dfs[name].shape[0]:>7,} rows  {dfs[name].shape[1]:>2} cols')

budget_functions    = dfs['budget_functions']
budget_subfunctions = dfs['budget_subfunctions']
functions_ref       = dfs['functions_ref']
subfunctions_ref    = dfs['subfunctions_ref']
agency              = dfs['agency']
federal_accounts    = dfs['federal_accounts']
geo_state           = dfs['geo_state']
geo_state_funding   = dfs['geo_state_funding']
geo_state_awarding  = dfs['geo_state_awarding']

for df in [budget_functions, budget_subfunctions, agency, federal_accounts,
           geo_state, geo_state_funding, geo_state_awarding]:
    if 'quarter' in df.columns:
        df['period'] = df['fy'] + (df['quarter'] - 1) / 4

print('\nAll 9 datasets loaded.')

Budget function and subfunction IDs are loaded as strings so leading zeros don't get dropped (e.g., `'050'` stays `'050'`, not `50`). The `period` column converts fiscal year + quarter into a single float like `2020.5` — this gives a clean x-axis when plotting time series across years.

In [ ]:
# Cell 4 — Data Quality Check
for name, df in dfs.items():
    nulls    = df.isnull().sum().sum()
    fy_range = f'FY{df["fy"].min()}–{df["fy"].max()}' if 'fy' in df.columns else ''
    print(f'{name:25}  {df.shape[0]:>7,} rows  {df.shape[1]:>2} cols  nulls={nulls}  {fy_range}')

A quick sanity check before any analysis. The null counts here should already be very low because the cleaning notebook handled them. If any file shows a surprisingly high null count, it means something in the cleaning step didn't fire correctly for that file.

In [ ]:
# Cell 5 — Summary Table
summary_rows = []
for name, df in dfs.items():
    summary_rows.append({
        'dataset'  : name,
        'rows'     : df.shape[0],
        'cols'     : df.shape[1],
        'nulls'    : int(df.isnull().sum().sum()),
        'memory_MB': round(df.memory_usage(deep=True).sum() / 1e6, 2),
    })
summary = pd.DataFrame(summary_rows)
print('Section 1 Complete — All datasets ready\n')
print(summary.to_string(index=False))

The memory column is worth keeping an eye on. Most files are small — budget functions and subfunctions are a few hundred KB at most. The agency geography files are a few MB. Nothing here is large enough to cause memory issues during EDA, so we can freely load everything at once without chunking.

---
## Section 2 — Budget Functions

In [ ]:
# Total federal spending — quarterly + annual
bf_valid = budget_functions.dropna(subset=['budget_function_id'])
total_ts = budget_functions.groupby(['fy', 'quarter', 'period'])['obligated_amount'].sum().reset_index()
annual   = budget_functions.groupby('fy')['obligated_amount'].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(total_ts['period'], total_ts['obligated_amount'], marker='o', ms=3, lw=1.5)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
axes[0].set_title('Total Federal Obligated Amount — Quarterly')
axes[0].set_xlabel('Fiscal Year')
axes[1].bar(annual.index, annual.values, color='steelblue')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
axes[1].set_title('Total Federal Obligated Amount — Annual')
axes[1].set_xlabel('Fiscal Year')
plt.tight_layout(); plt.show()

This is the highest-level view — total federal obligations across all budget functions by quarter and year. The quarterly chart makes it easy to spot the within-year rhythm: Q4 (July–September) almost always spikes because agencies rush to spend before the fiscal year closes and unspent funds get reclaimed. The annual bar chart tells the bigger story — spending grew steadily pre-pandemic, spiked hard in 2020 and 2021 when COVID relief hit, and has been finding a new (higher) normal since.

In [ ]:
# Top 10 functions — quarterly time series
top10_ids  = bf_valid.groupby('budget_function_id')['obligated_amount'].sum().nlargest(10).index.tolist()
func_label = bf_valid.drop_duplicates('budget_function_id').set_index('budget_function_id')['budget_function_name'].to_dict()

fig, ax = plt.subplots(figsize=(16, 6))
for fid in top10_ids:
    sub = bf_valid[bf_valid['budget_function_id'] == fid].sort_values('period')
    ax.plot(sub['period'], sub['obligated_amount'], marker='o', ms=2, lw=1.5,
            label=f'{fid} {func_label[fid][:28]}')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Top 10 Budget Functions — Quarterly Spending')
ax.set_xlabel('Fiscal Year')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()

Social Security, Medicare, and Income Security are the three lines that dominate this chart. Social Security and Medicare are steady climbers — demographics drive them, not politics. Income Security is the volatile one: it sits quietly below the others for years, then in 2020-2021 it shoots up sharply. That's the COVID relief money — stimulus payments, expanded unemployment, pandemic assistance — all landing in this one bucket. After 2022 it collapses back down just as fast. National Defense is the most visually stable line across all years.

In [ ]:
# Spending share (latest FY) + YoY growth (top 5)
latest_fy   = bf_valid['fy'].max()
share       = (bf_valid[bf_valid['fy'] == latest_fy]
                 .groupby('budget_function_name')['obligated_amount'].sum()
                 .sort_values(ascending=False))
annual_func = (bf_valid.groupby(['fy', 'budget_function_id', 'budget_function_name'])['obligated_amount']
                       .sum().reset_index().sort_values(['budget_function_id', 'fy']))
annual_func['yoy_pct'] = annual_func.groupby('budget_function_id')['obligated_amount'].pct_change() * 100
top5_ids    = annual_func.groupby('budget_function_id')['obligated_amount'].sum().nlargest(5).index.tolist()
yoy_pivot   = annual_func[annual_func['budget_function_id'].isin(top5_ids)].pivot(
    index='fy', columns='budget_function_name', values='yoy_pct')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(share.index[::-1], share.values[::-1], color=plt.cm.tab20.colors[:len(share)])
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(trillions))
axes[0].set_title(f'Budget Function Spending Share — FY{latest_fy}')
yoy_pivot.plot(ax=axes[1], marker='o', ms=4, lw=1.5)
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_title('YoY Growth % — Top 5 Budget Functions')
axes[1].set_ylabel('YoY Growth (%)')
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

The share chart shows the latest fiscal year breakdown — at a glance, this tells you which functions are eating the biggest slice of the budget right now. The YoY growth chart on the right is where things get interesting for forecasting: Income Security shows the most dramatic swings (positive in 2020-2021, sharply negative in 2022 as relief programs wound down). Functions that stay close to the zero line year after year — like Defense and Social Security — are the easiest to forecast. The ones with large swings need models that can handle structural breaks.

In [ ]:
# Quarterly seasonality
season = bf_valid.groupby('quarter')['obligated_amount'].mean()
qdata  = [bf_valid[bf_valid['quarter'] == q]['obligated_amount'].values for q in [1,2,3,4]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(['Q1','Q2','Q3','Q4'], season.values, color='steelblue')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
axes[0].set_title('Average Spending by Quarter (all functions, all years)')
axes[1].boxplot(qdata, labels=['Q1','Q2','Q3','Q4'], showfliers=False)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(billions))
axes[1].set_title('Spending Distribution by Quarter')
plt.tight_layout(); plt.show()

This confirms the Q4 effect clearly. On average, the federal government spends more in Q4 than in any other quarter — and the box plot shows this holds true across most budget functions, not just a few. Q1 tends to be the lightest quarter. This seasonality pattern is consistent enough that Prophet and SARIMA models should pick it up automatically, but it's worth verifying it's present in the training data before fitting.

---
## Section 3 — Budget Subfunctions

In [ ]:
# Remove contaminated rows (subfunction_id == function_id — API quirk)
bsf_clean    = budget_subfunctions[budget_subfunctions['budget_subfunction_id'] != budget_subfunctions['budget_function_id']].copy()
contaminated = len(budget_subfunctions) - len(bsf_clean)
print(f'Subfunctions: {len(budget_subfunctions):,} total  |  {contaminated} contaminated removed  |  {len(bsf_clean):,} clean')
print(f'Unique subfunctions: {bsf_clean["budget_subfunction_id"].nunique()}')

top15_sf = (bsf_clean.groupby(['budget_subfunction_id', 'budget_subfunction_name'])['obligated_amount']
                     .sum().reset_index().nlargest(15, 'obligated_amount'))
top15_sf['label'] = top15_sf['budget_subfunction_id'] + ' ' + top15_sf['budget_subfunction_name'].str[:35]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(top15_sf['label'][::-1].values, top15_sf['obligated_amount'][::-1].values)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Top 15 Budget Subfunctions — Total Spend (All Years)')
plt.tight_layout(); plt.show()

The USASpending API sometimes returns the parent function as a row inside the subfunctions response — those contaminated rows have the same ID for both function and subfunction, and they're removed here to avoid double-counting. The bar chart shows that Social Security retirement benefits and Medicare hospital insurance sit at the very top by a wide margin. These two alone account for a massive share of all federal spending and have barely any volatility — they're the most predictable series in the entire dataset.

In [ ]:
# Top 8 subfunctions — quarterly time series
top8_sf_ids = top15_sf.nlargest(8, 'obligated_amount')['budget_subfunction_id'].tolist()
sf_label    = bsf_clean.drop_duplicates('budget_subfunction_id').set_index('budget_subfunction_id')['budget_subfunction_name'].to_dict()

fig, ax = plt.subplots(figsize=(16, 6))
for sfid in top8_sf_ids:
    sub = bsf_clean[bsf_clean['budget_subfunction_id'] == sfid].sort_values('period')
    ax.plot(sub['period'], sub['obligated_amount'], marker='o', ms=2, lw=1.5,
            label=f'{sfid} {sf_label.get(sfid, "")[:28]}')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Top 8 Budget Subfunctions — Quarterly Spending')
ax.set_xlabel('Fiscal Year')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()

The time series at the subfunction level shows the same COVID spike pattern, but now you can see exactly which subfunctions caused it — unemployment compensation and other income support programs within the Income Security function. One subfunction to watch for forecasting: interest on the public debt. It barely registers before 2022, then starts climbing as interest rates went up. Unlike most other series, interest payments don't respond to discretionary policy — they're driven purely by existing debt levels and market rates, so they need a different modeling approach.

---
## Section 4 — Agency Spending

In [ ]:
# Agency overview + total quarterly time series
print(f'Agency rows: {len(agency):,}   FY: {agency["fy"].min()}–{agency["fy"].max()}')
print(f'Unique agencies: {agency["agency_id"].nunique()}')
print(f'Agency types: {agency["agency_type"].value_counts().to_dict()}')
print(f'Negative amounts: {(agency["obligated_amount"] < 0).sum()}')

top15_ag = (agency.groupby(['agency_id', 'agency_name'])['obligated_amount']
                  .sum().reset_index().nlargest(15, 'obligated_amount'))
top15_ag['total_T'] = top15_ag['obligated_amount'] / 1e12
print('\nTop 15 agencies:')
print(top15_ag[['agency_name','total_T']].to_string(index=False))

ag_total = agency.groupby(['fy','quarter','period'])['obligated_amount'].sum().reset_index()
fig, ax  = plt.subplots(figsize=(14, 5))
ax.plot(ag_total['period'], ag_total['obligated_amount'], marker='o', ms=3, lw=1.5, color='steelblue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Total Agency Obligated Amount — Quarterly')
ax.set_xlabel('Fiscal Year')
plt.tight_layout(); plt.show()

The negative amounts are legitimate — agencies record negative obligations when contracts are cancelled, funds are returned, or prior-year estimates get revised downward. They shouldn't be filtered out. The total quarterly chart mirrors the budget function view but now from the agency side — the same COVID spike shows up in 2020-2021. The top 15 table is useful for deciding which agencies to build individual forecasts for: SSA and HHS dominate by a large margin, followed by Defense and Treasury.

In [ ]:
# Top 8 agencies — quarterly time series
top8_ag_ids = top15_ag.nlargest(8, 'obligated_amount')['agency_id'].tolist()
ag_label    = agency.drop_duplicates('agency_id').set_index('agency_id')['agency_name'].to_dict()

fig, ax = plt.subplots(figsize=(16, 6))
for aid in top8_ag_ids:
    sub = agency[agency['agency_id'] == aid].sort_values('period')
    ax.plot(sub['period'], sub['obligated_amount'], marker='o', ms=2, lw=1.5,
            label=ag_label.get(aid, str(aid))[:40])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Top 8 Agencies — Quarterly Spending')
ax.set_xlabel('Fiscal Year')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()

This chart separates agencies that are steady from those that move around. SSA barely changes quarter to quarter — its spending is driven by the number of beneficiaries, which shifts slowly. The Department of the Treasury is the most volatile line on this chart: it spiked sharply in 2020-2021 because Treasury was the distribution channel for COVID stimulus payments and then dropped back. That kind of external-shock volatility is difficult for time series models unless the shock can be modeled as a known intervention.

In [ ]:
# Agency × FY heatmap (top 20)
top20_ag_ids = agency.groupby('agency_id')['obligated_amount'].sum().nlargest(20).index.tolist()
ag_pivot     = (agency[agency['agency_id'].isin(top20_ag_ids)]
                  .groupby(['agency_name','fy'])['obligated_amount'].sum()
                  .unstack('fy'))

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(ag_pivot / 1e9, ax=ax, cmap='YlOrRd', fmt='.0f', annot=True,
            linewidths=0.3, annot_kws={'size': 6}, cbar_kws={'label': 'Obligated $B'})
ax.set_title('Top 20 Agencies — Annual Spending ($B)')
ax.set_xlabel('Fiscal Year'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

The heatmap gives you a full picture of all 20 agencies across all years at once. Darker cells mean more spending. Two things stand out immediately: first, Treasury lights up brightly in 2020-2021 and then goes dark again — the COVID effect is unmistakable. Second, some agencies have gaps or near-zero cells for certain years, which could mean data wasn't collected for that period or the agency genuinely had near-zero obligations. Those cells need to be checked before training any agency-level forecasting model, because a missing year looks exactly like a spending crash to a model that doesn't know better.

---
## Section 5 — Federal Accounts

In [ ]:
# Federal accounts overview + top 15 by total spend
print(f'Federal accounts rows: {len(federal_accounts):,}   FY: {federal_accounts["fy"].min()}–{federal_accounts["fy"].max()}')
print(f'Unique federal accounts: {federal_accounts["federal_account_id"].nunique()}')
print(f'Unique budget functions:  {federal_accounts["budget_function_id"].nunique()}')
print(f'Negative amounts: {(federal_accounts["obligated_amount"] < 0).sum()} ({(federal_accounts["obligated_amount"] < 0).mean()*100:.1f}%)')

top15_fa = (federal_accounts.groupby(['federal_account_id', 'federal_account_name'])['obligated_amount']
                             .sum().reset_index().nlargest(15, 'obligated_amount'))
top15_fa['total_T'] = top15_fa['obligated_amount'] / 1e12
print('\nTop 15 federal accounts by total spend:')
print(top15_fa[['federal_account_name', 'total_T']].to_string(index=False))

Federal accounts sit one level below budget functions — each account belongs to a specific function and is owned by a specific agency. With 2,236 unique accounts, most of them are small. The top 15 accounts by total spend account for a disproportionate share of all federal obligations, which is a classic long-tail distribution. The negative amounts are legitimate deobligations — money that was committed but later cancelled or returned. At the account level these are more common than at the function level because contracts get modified or terminated at this granularity.

In [ ]:
# Distribution of account spending + top 15 bar + time series of top 8
fa_annual = federal_accounts.groupby(['federal_account_id', 'federal_account_name', 'fy'])['obligated_amount'].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution (log scale)
fa_pos = fa_annual[fa_annual['obligated_amount'] > 0]['obligated_amount']
axes[0].hist(np.log10(fa_pos + 1), bins=40, edgecolor='white', lw=0.3, color='steelblue')
axes[0].set_xlabel('log10(Annual Obligated Amount)')
axes[0].set_title('Distribution of Federal Account Spending (log scale)')
axes[0].set_ylabel('Number of accounts')

# Top 15 bar
top15_labels = [n[:40] for n in top15_fa['federal_account_name']]
axes[1].barh(top15_labels[::-1], top15_fa['obligated_amount'][::-1].values)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(trillions))
axes[1].set_title('Top 15 Federal Accounts — Total Spend (All Years)')
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout(); plt.show()

The log-scale distribution shows the classic long tail — most federal accounts spend very little, and a small number account for almost everything. This matters for forecasting: individual models (Prophet, SARIMA) will only be worth building for the top accounts. For the rest, a single XGBoost model trained across all accounts is more practical. The top 15 bar chart identifies which accounts those are — Social Security retirement, Medicare trust funds, and defense procurement dominate.

In [ ]:
# Top 8 federal accounts — quarterly time series
top8_fa_ids = top15_fa.nlargest(8, 'obligated_amount')['federal_account_id'].tolist()
fa_label    = federal_accounts.drop_duplicates('federal_account_id').set_index('federal_account_id')['federal_account_name'].to_dict()
federal_accounts['period'] = federal_accounts['fy'] + (federal_accounts['quarter'] - 1) / 4

fig, ax = plt.subplots(figsize=(16, 6))
for faid in top8_fa_ids:
    sub = federal_accounts[federal_accounts['federal_account_id'] == faid].sort_values('period')
    ax.plot(sub['period'], sub['obligated_amount'], marker='o', ms=2, lw=1.5,
            label=fa_label.get(faid, str(faid))[:45])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Top 8 Federal Accounts — Quarterly Spending')
ax.set_xlabel('Fiscal Year')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()

At the account level, the time series are much more volatile than at the function level — that's expected, because account-level spending responds to specific program changes, contract awards, and legislative decisions that smooth out when aggregated upward. The Social Security and Medicare accounts are the exception: they're smooth and predictable even at this granular level because they're driven by beneficiary counts, not discretionary decisions. The COVID spike shows up in a handful of accounts that were used to distribute relief funds — those will need the COVID anomaly flag when we build forecasting models.

In [ ]:
# Spending by budget function — stacked bar (top 8 functions via federal accounts)
bf_names    = bf_valid.drop_duplicates('budget_function_id').set_index('budget_function_id')['budget_function_name'].to_dict()
fa_func     = federal_accounts.groupby(['fy', 'budget_function_id'])['obligated_amount'].sum().reset_index()
top8_f_ids  = fa_func.groupby('budget_function_id')['obligated_amount'].sum().nlargest(8).index.tolist()
fa_func_top = fa_func[fa_func['budget_function_id'].isin(top8_f_ids)]
pivot       = fa_func_top.pivot(index='fy', columns='budget_function_id', values='obligated_amount')
pivot.columns = [f'{c} {bf_names.get(c,"")[:18]}' for c in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(kind='bar', ax=ax, stacked=True)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Federal Account Spending by Budget Function — Annual (Stacked)')
ax.set_xlabel('Fiscal Year')
ax.legend(fontsize=7, loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

This stacked bar shows how the federal account totals roll up to budget functions year by year. It's the same story as before — Social Security and Medicare are the stable base layers, Income Security jumped sharply in 2020-2021 and then contracted, and Net Interest has been growing since 2022. The value of this chart for the modeling phase is the join validation: the totals here should match the budget function quarterly totals from Section 2 within rounding. If they don't, there's a data consistency issue between the two files that needs to be resolved before building hierarchical forecasts.

---
## Section 6 — Geography (State Level)

In [ ]:
# State overview + top 10 annual time series
print(f'Geo state rows: {len(geo_state):,}   FY: {geo_state["fy"].min()}–{geo_state["fy"].max()}')
print(f'Unique states/territories: {geo_state["geo_code"].nunique()}')

top_states  = (geo_state.groupby(['geo_code','geo_name'])['obligated_amount']
                        .sum().reset_index().nlargest(15, 'obligated_amount'))
top_states['total_T'] = top_states['obligated_amount'] / 1e12
print('\nTop 15 states:')
print(top_states[['geo_code','geo_name','total_T']].to_string(index=False))

top10_codes = top_states.nlargest(10, 'obligated_amount')['geo_code'].tolist()
geo_annual  = geo_state.groupby(['fy','geo_code','geo_name'])['obligated_amount'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for code in top10_codes:
    sub  = geo_annual[geo_annual['geo_code'] == code].sort_values('fy')
    name = sub['geo_name'].iloc[0] if len(sub) else code
    ax.plot(sub['fy'], sub['obligated_amount'], marker='o', ms=3, lw=1.5, label=name)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Top 10 States — Annual Federal Spending')
ax.set_xlabel('Fiscal Year')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

California and Texas top the absolute spending list mostly because of population size — more people means more Social Security, Medicare, and unemployment payments flowing to those states. Virginia is the interesting outlier: it consistently ranks near the top despite not being one of the most populous states. The reason is Northern Virginia — the Pentagon, defense contractors, and the dense cluster of federal agencies in that region pull in an enormous amount of federal money. The annual time series shows all top states spiked in 2020-2021, but they didn't all spike equally — states with larger Medicaid programs and higher unemployment insurance payouts saw bigger jumps.

In [ ]:
# Per capita spending + quarterly pattern
geo_pop    = geo_state.dropna(subset=['population'])
latest_pop = geo_pop['fy'].max()
geo_pc     = (geo_pop[geo_pop['fy'] == latest_pop]
                .groupby(['geo_code','geo_name'])[['obligated_amount','population']].sum().reset_index())
geo_pc['per_capita'] = geo_pc['obligated_amount'] / geo_pc['population']
geo_pc = geo_pc[geo_pc['population'] > 0].nlargest(20, 'per_capita')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(geo_pc['geo_name'][::-1].values, geo_pc['per_capita'][::-1].values)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_title(f'Federal Spending Per Capita — FY{latest_pop}')

top5_codes = top_states.nlargest(5, 'obligated_amount')['geo_code'].tolist()
geo_q      = geo_state[geo_state['geo_code'].isin(top5_codes)].sort_values('period')
for code in top5_codes:
    sub  = geo_q[geo_q['geo_code'] == code]
    name = sub['geo_name'].iloc[0] if len(sub) else code
    axes[1].plot(sub['period'], sub['obligated_amount'], marker='o', ms=2, lw=1.5, label=name)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(billions))
axes[1].set_title('Top 5 States — Quarterly Federal Spending')
axes[1].set_xlabel('Fiscal Year')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

The per capita chart flips the rankings completely. DC sits at the very top by a huge margin — not because DC residents receive the most benefits, but because federal agency headquarters, grant disbursements, and contracting activity gets attributed to DC zip codes regardless of where the actual work happens. Maryland and Virginia also show up high for the same reason — federal installations, contractors, and agency campuses. Alaska tends to rank high per capita because it receives outsized federal transfers relative to its small population: military installations, infrastructure programs, and resource-related payments. The quarterly chart on the right shows the within-year spending rhythm for the top 5 states — you can see the Q4 spike pattern repeating consistently across all of them.

---
## Section 7 — Summary

In [ ]:
print('=== EDA Summary ===')
print(f'Budget Functions:    {bf_valid["budget_function_id"].nunique()} functions  |  FY{bf_valid["fy"].min()}–{bf_valid["fy"].max()}  |  {len(bf_valid):,} rows')
print(f'Budget Subfunctions: {bsf_clean["budget_subfunction_id"].nunique()} subfunctions  |  {len(bsf_clean):,} rows')
print(f'Agency:              {agency["agency_id"].nunique()} agencies  |  FY{agency["fy"].min()}–{agency["fy"].max()}  |  {len(agency):,} rows')
print(f'Federal Accounts:    {federal_accounts["federal_account_id"].nunique()} accounts  |  {len(federal_accounts):,} rows')
print(f'Geography (State):   {geo_state["geo_code"].nunique()} states/territories  |  FY{geo_state["fy"].min()}–{geo_state["fy"].max()}  |  {len(geo_state):,} rows')
print()
print('Recommended forecasting targets:')
print('  1. Total federal spending (budget_functions_quarterly) — strong Q4 seasonality')
print('  2. Per-function spending — trend + seasonality vary by function')
print('  3. Agency-level spending — SSA and HHS most predictable, Treasury most volatile')
print('  4. State-level geographic spending — strong geographic concentration')

A few things worth carrying forward into the forecasting phase. The Q4 seasonality pattern is real and consistent — any model that ignores it will systematically underpredict Q4 and overpredict Q1. The COVID years (2020-2021) are structural breaks, not noise — models trained on pre-2020 data will need those years treated as known interventions or the out-of-sample forecasts will be badly off. The most forecastable series are the mandatory spending programs: Social Security retirement, Medicare, and Defense. The hardest are the ones driven by emergency legislation, because by definition those aren't predictable from historical patterns alone.